# Avian Influenza Dataset Pipeline

Author: Alexander Maksiaev

Purpose: Create weekly dataset using NCBI Virus, Andersen Lab, and GISAID.

Notes: 
* This script only works in a Linux environment with bioconda installed. 
* The directory where this script is housed should also house "utils.py". 
* All data from GISAID must be downloaded prior to running this script.

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
from itertools import islice
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [ ]:
# Directory paths and input

# Input
# browser = input("Browser (Firefox, Chrome, or Edge): ")
# sleep_time = input("Seconds to wait in between clicks (recommended 5): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date (format: MM-DD-YYYY): ")
# prev_end_date = input("End date of previous dataset (format: MM-DD-YYYY): ")
# serotype = input("Serotype (e.g. H5N1): ")
# serotypes = list(serotype)
# genotypes = input("Genotypes (separate with commas and no spaces in between genotypes): ")
# genotypes = genotypes.split(",")


# Dates and locations
browser = "Firefox"
sleep_time = "6"
locations = "Antarctica,North America,South America"
start_date = "11-01-2021"
end_date = "02-06-2026"
prev_end_date = "01-09-2026"
date_range = start_date + "--" + end_date
prev_date_range = start_date + "--" + prev_end_date

# Maintenance serotypes and genotypes
serotypes = ["H5N1"]
genotypes = ["B3.13", "D1.1", "D1.3"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"

# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
# references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"

home = "/data/maksiaevai/"
downloads = "/home/maksiaevai/Downloads/"
references = "/data/maksiaevai/Avian_Flu/references/"

# downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 
prev_downloads_saved = home + prev_date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 


complete_files = home + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
andersen = home + "avian-influenza/metadata/"
ncbi_virus = complete_files + "NCBI_Virus/"
gisaid = complete_files + "GISAID/"

for folder in [complete_files, ncbi_virus, gisaid]:
    if not os.path.exists(folder): # checking if the directory exists or not
        os.makedirs(folder) # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# All serotypes and genotypes
# serotype = ""
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])


## Downloading Data

In [ ]:
os.chdir(downloads)

ncbi_virus_downloads = ncbi_virus + "Downloads/"
if not os.path.exists(ncbi_virus_downloads): # checking if the directory exists or not
    os.makedirs(ncbi_virus_downloads) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(ncbi_virus_downloads):
    if len(files) != 0:
        break 
    else: # If we don't have any downloaded files
        # Get files
        open_ncbi_virus(browser, sleep_time, locations, start_date, end_date) # Will not work on BioWulf because of permissions. Sorry

        # Re-try 
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(ncbi_virus_downloads, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    break 

WebDriverException: Message: <!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">
<html><head>
<meta type="copyright" content="Copyright (C) 1996-2021 The Squid Software Foundation and contributors">
<meta http-equiv="Content-Type" content="text/html; charset=utf-8">
<title>ERROR: The requested URL could not be retrieved</title>
<style type="text/css"><!--
 /*
 * Copyright (C) 1996-2021 The Squid Software Foundation and contributors
 *
 * Squid software is distributed under GPLv2+ license and includes
 * contributions from numerous individuals and organizations.
 * Please see the COPYING and CONTRIBUTORS files for details.
 */

/*
 Stylesheet for Squid Error pages
 Adapted from design by Free CSS Templates
 http://www.freecsstemplates.org
 Released for free under a Creative Commons Attribution 2.5 License
*/

/* Page basics */
* {
	font-family: verdana, sans-serif;
}

html body {
	margin: 0;
	padding: 0;
	background: #efefef;
	font-size: 12px;
	col